# System 1 Models in Safety Research: workshop notebook

CE 7393 guest lecture, 14 October 2026. Amir Rafe, Ph.D., AIT Lab, Texas State University.

This notebook runs the open-source System 1 model **Laya** (Apache 2.0, about 0.4 billion parameters) on a free Colab CPU. No API key is needed. Every report in this notebook is **synthetic**.

Run the cells in order. Cell 1 takes one to two minutes the first time because it installs Laya and downloads the checkpoint.

## Cell 1 · Install and load (Lab 1)

In [ ]:
!pip -q install laya==0.4.2
from laya import Router
router = Router()   # downloads the open checkpoint once

## Cell 2 · A helper that prints probabilities as bars

In [ ]:
def show(result):
    for qid, a in result["answers"].items():
        if "noul" in a:
            p = a["noul"]
            print(f"{qid:<22} P(yes) {p:4.2f}  " + "#" * round(p * 20))
        else:
            probs = a.get("probabilities", {})
            top = max(probs, key=probs.get) if probs else a.get("choice")
            print(f"{qid:<22} top option: {top} ({probs.get(top, 0):.2f})")

## Cell 3 · One call: state plus typed questions in, probabilities out (Lab 1)

Read the report first. Which answers do you expect? Then run the cell.

In [ ]:
state = ("Worker on level 3 scaffold. East guardrail missing. "
         "Worker slipped, harness held, no injury.")
questions = {
  "fall_hazard":  {"type": "noul", "instructions": "Does the report describe a fall hazard?"},
  "harness_held": {"type": "noul", "instructions": "Did fall protection stop the fall?"},
  "injury":       {"type": "noul", "instructions": "Was anyone injured?"},
}
result = router.predict(state, questions)
show(result)

**Try this.** Add plain yes and no descriptions to each question and run again. Do the answers move? In our benchmark (Paper 4A), swapping the words yes and no in these descriptions flipped 91 of 100 answers for this checkpoint.

In [ ]:
def yes_no(instr, yes, no):
    return {"type": "noul", "instructions": instr, "criteria": {"true": yes, "false": no}}

questions_with_words = {
  "fall_hazard":  yes_no("Does the report describe a fall hazard?",
                         "The report describes a hazard that could cause a fall from height.",
                         "The report describes no fall hazard."),
  "harness_held": yes_no("Did fall protection stop the fall?",
                         "A harness or other fall protection stopped the fall.",
                         "Fall protection did not stop a fall, or none is mentioned."),
  "injury":       yes_no("Was anyone injured?",
                         "The report says a person was injured.",
                         "The report says no one was injured, or mentions no injury."),
}
show(router.predict(state, questions_with_words))

## Cell 4 · A gated schema (Lab 2)

The follow-up question runs only when the screen question says yes with probability above 0.5. Add your own yes or no question to `my_questions`.

In [ ]:
report = ("Unit 2 was stopped at the light on FM 1626. Driver of Unit 1 stated he felt dizzy "
          "and blacked out before the crash. Witness stated Unit 1 did not brake. "
          "Driver of Unit 1 transported to hospital by EMS.")

screen = {"medical_episode": {"type": "noul", "instructions":
          "The narrative states that a driver had a medical episode, such as fainting, "
          "a seizure, or a diabetic event, before or during the crash."}}
follow_up = {"medical_type": {"type": "choice",
             "instructions": "What kind of medical episode does the narrative describe?",
             "criteria": {"seizure": "A seizure or convulsion.",
                          "loss_of_consciousness": "Fainting, blacking out, or passing out.",
                          "diabetic": "Low or high blood sugar.",
                          "cardiac": "Heart attack or chest pain.",
                          "other": "Another medical event."}}}
my_questions = {
  "witness_cited": {"type": "noul", "instructions": "The narrative cites a statement from a witness."},
  # "my_question": {"type": "noul", "instructions": "Write your own yes or no question here."},
}

first = router.predict(report, {**screen, **my_questions})
show(first)
if first["answers"]["medical_episode"]["noul"] > 0.5:      # the gate
    print("\ngate open, asking the follow-up:")
    show(router.predict(report, follow_up))
else:
    print("\ngate shut, follow-up skipped")

## Cell 5 · Measure calibration, then fix it (Lab 3)

A probability is useful only if 0.9 means right nine times in ten. You check that against human labels. The data below are **simulated** to behave like an over-confident model; with real work you would use your own labeled reports.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(7)
truth_p = rng.uniform(0.05, 0.95, 2000)          # how likely each report really is "yes"
y = rng.binomial(1, truth_p)                      # human labels
logit = np.log(truth_p / (1 - truth_p))
p = 1 / (1 + np.exp(-2.5 * logit))                # an over-confident model's probabilities

def reliability(p, y, bins=5):
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    ece = 0
    for b in range(bins):
        m = idx == b
        conf, acc = p[m].mean(), y[m].mean()
        ece += m.mean() * abs(conf - acc)
        print(f"stated {conf:4.2f}   observed {acc:4.2f}   n={m.sum()}")
    print(f"expected calibration error {ece:.3f}")

print("as returned"); reliability(p, y)

half = len(p) // 2                                 # fit on one half, check on the other
pc = np.clip(p, 1e-4, 1 - 1e-4)
z = np.log(pc / (1 - pc)).reshape(-1, 1)
platt = LogisticRegression().fit(z[:half], y[:half])
p_new = platt.predict_proba(z[half:])[:, 1]
print("\nafter Platt scaling, held out"); reliability(p_new, y[half:])

## Cell 6 · Your domain (Lab 6)

Four synthetic reports from construction, bridges, water and buildings. Change the questions or paste a report from your own field.

In [ ]:
reports = {
  "trench": ("Excavation at Pier 3 is about 8 feet deep with no shoring or trench box. Laborer entered "
             "the trench to clear a pump hose. Spoil pile placed at the trench edge. Foreman stopped work.",
             "Does the report describe a trench cave-in hazard?"),
  "bridge": ("Span 2, girder G3. Diagonal crack about 0.5 mm wide near the east bearing. Rust staining "
             "below the deck joint. No section loss observed. Bearing appears frozen.",
             "Does the note describe a defect that needs a site visit before the next routine inspection?"),
  "water":  ("Customers on Elm Street report low pressure and brown water since 6 AM. Crew found a 12-inch "
             "cast iron main break near hydrant 44. Roadway surface is undermined and sagging.",
             "Does the report describe a hazard to traffic on the roadway?"),
  "fire":   ("Fire started in a second-floor kitchen of a four-story apartment building. Corridor smoke "
             "detectors activated. The building has no sprinkler system. One resident treated for smoke inhalation.",
             "Was a fire suppression system missing or did it fail?"),
}
for name, (text, q) in reports.items():
    print(f"--- {name}")
    show(router.predict(text, {"q": {"type": "noul", "instructions": q}}))

## What to take away

1. A System 1 model returns probabilities for typed questions, not text.
2. The checkpoint and the wording of the options both change the answers. Test the model you plan to use on your own reports.
3. Measure calibration on held-out human labels before a probability carries a decision, and recalibrate when it is off.
4. These models read what people wrote. When physics has to be squeezed into words, a simple model on the same words often does better (Paper 3).